# P2 official final submission

## Goal

This notebook is a self-contained, top-to-bottom materializer and validator for the frozen clean-lineage P2 candidate. It reads only organizer-distributed inputs from the configured data directory and package-local model or frozen inference assets. It never opens hidden truth and never uploads.

Official metric: `pooled RMSE (C)`. Exact mode uses the frozen output of the scratch-trained three-fit v52 ensemble; full training source is bundled for audit.

## Setup

Run this notebook with the working directory set to this problem package.

In [ ]:
from pathlib import Path
import os
import sys

PACKAGE_DIR = Path.cwd().resolve()
if not (PACKAGE_DIR / 'contract.json').is_file():
    raise RuntimeError('Run from the packaged problem directory containing contract.json')
raw_data_dir = os.environ.get('P2_DATA_DIR')
if not raw_data_dir:
    raise RuntimeError('P2_DATA_DIR must point to the organizer-distributed P2 directory')
DATA_DIR = Path(raw_data_dir).expanduser().resolve()
sys.path.insert(0, str(PACKAGE_DIR))
from common import bounded_receipt
import run_submission

print({'package': PACKAGE_DIR.name, 'data_dir_present': DATA_DIR.is_dir()})

## Step 1 — hash-bound preflight

Only bounded metadata is displayed; no prediction row is printed.

In [ ]:
preflight = run_submission.preflight(DATA_DIR, PACKAGE_DIR)
bounded_receipt(preflight)

## Step 2 — exact local materialization

In [ ]:
output_path = PACKAGE_DIR / 'outputs' / 'P2_submission.csv'
receipt = run_submission.materialize(DATA_DIR, PACKAGE_DIR, output_path)
bounded_receipt(receipt)

## Checks

The schema, row count, key order, finite/domain rules and frozen SHA-256 are fail-closed.

In [ ]:
assert receipt['status'] == 'READY_EXACT_NOT_UPLOADED'
assert receipt['candidate_hash_exact']
assert receipt['key_order_exact']
assert receipt['package_atomic']
assert output_path.is_file()
print({'status': receipt['status'], 'rows': receipt['rows'], 'sha256': receipt['sha256']})

## Next steps

Keep the generated CSV beside its receipt. Before the website action, compare the title and one-line summary in `contract.json`, then upload deliberately. This notebook performs no network action.